In [10]:
%pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

In [14]:
from langchain_core.tools import tool



In [12]:
# step 1: create a function that you want to use as a tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

In [15]:
# step 2: create a Tool object that wraps the function



@tool 
def add_numbers_tool(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


In [17]:
result = add_numbers_tool.invoke({"a": 3, "b": 5})
print(result)

8


In [19]:
print(add_numbers_tool.name)
print(add_numbers_tool.description)
print(add_numbers_tool.args)

add_numbers_tool
Add two numbers together.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


Using StructuredTool

What is StructuredTool?

StructuredTool is an advanced version of a Tool in LangChain that allows:

Multiple inputs

Typed arguments

Input validation

Schema-based structure

Simple definition:

StructuredTool = A tool that uses a structured input schema (usually Pydantic) to handle multiple validated arguments.

In [21]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field


class AddNumbersInput(BaseModel):
    a: int = Field(..., description="The first number to add")
    b: int = Field(..., description="The second number to add")

In [22]:
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

In [23]:
add_numbers_tool=StructuredTool.from_function(
    func=add_numbers,
    name="add_numbers",
    description="Add two numbers together.",
    
    args_schema=AddNumbersInput)    

In [24]:
result = add_numbers_tool.invoke({"a": 2, "b": 3})
print(result)
print(add_numbers_tool.name)
print(add_numbers_tool.description)
print(add_numbers_tool.args)

5
add_numbers
Add two numbers together.
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


# Using basetool class 


What is BaseTool?

BaseTool is the abstract base class for all tools in LangChain.

It defines the core structure and behavior that every tool must follow.

Simple definition:

BaseTool = The parent class that provides the foundation for all tools in LangChain.

In [49]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type


# 1️⃣ Input Schema (Must inherit from BaseModel)
class AddNumbersInput(BaseModel):
    a: int = Field(..., description="First number")
    b: int = Field(..., description="Second number")


# 2️⃣ Tool Class (Must inherit from BaseTool)
class AddNumbersTool(BaseTool):
    name: str = "add_numbers"
    description: str = "Add two numbers together."

    args_schema: Type[BaseModel] = AddNumbersInput

    # IMPORTANT: self must be present
    def _run(self, a: int, b: int) -> int:
        return a + b

    async def _arun(self, a: int, b: int) -> int:
        raise NotImplementedError("Async not implemented")


# 3️⃣ Use Tool
tool = AddNumbersTool()

result = tool.invoke({"a": 4, "b": 6})
print(result)

10


# Toolkit


In [50]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [51]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [52]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
